In [1]:
import torch 
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from pathlib import Path

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Load image 
image = load_image('closing_disclosure.webp')

In [8]:
# Initialize processor and model
processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
model = AutoModelForVision2Seq.from_pretrained(
    "ds4sd/SmolDocling-256M-preview",
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
).to(DEVICE)


In [9]:
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    },
]

In [10]:
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [11]:
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

In [12]:
# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

<doctag><section_header_level_1><loc_27><loc_29><loc_155><loc_44>Closing Disclosure</section_header_level_1>
<text><loc_29><loc_52><loc_100><loc_59>Closing Information</text>
<text><loc_29><loc_61><loc_61><loc_67>Date Issued</text>
<text><loc_88><loc_61><loc_122><loc_67>4/15/2013</text>
<text><loc_29><loc_69><loc_67><loc_75>Closing Date</text>
<text><loc_88><loc_69><loc_122><loc_75>4/15/2013</text>
<text><loc_29><loc_77><loc_83><loc_83>Disbursement Date</text>
<text><loc_88><loc_77><loc_122><loc_83>4/15/2013</text>
<text><loc_29><loc_85><loc_83><loc_91>Settlement Agent</text>
<text><loc_88><loc_85><loc_137><loc_91>Epsilon Title Co.</text>
<text><loc_29><loc_93><loc_47><loc_99>File #</text>
<text><loc_88><loc_93><loc_117><loc_99>12-3456</text>
<text><loc_29><loc_100><loc_56><loc_106>Property</text>
<text><loc_88><loc_100><loc_155><loc_106>456 Somewhere Ave</text>
<text><loc_29><loc_110><loc_57><loc_116>Sale Price</text>
<text><loc_88><loc_100><loc_155><loc_106>Anytown, ST 12345</text>
<

In [13]:
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")


In [14]:
print(doc.export_to_markdown())

## Closing Disclosure

Closing Information

Date Issued

4/15/2013

Closing Date

4/15/2013

Disbursement Date

4/15/2013

Settlement Agent

Epsilon Title Co.

File #

12-3456

Property

456 Somewhere Ave

Sale Price

Anytown, ST 12345

Transaction Information

Borrower

Michael Jones and Mary Stone

123 Anywhere Street

Anytown, ST 12345

Steve Cole and Amy Doe

321 Somewhere Drive

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Anytown, ST 12345

Sale Price

$180,000

Loan Terms

Can this amount increase after closing?

Loan Amount

$162,000

NO

Interest Rate

3.875%

NO

Monthly Principal &amp; Interest

$761.78

NO

See Projected Payments below for your Estimated Total Monthly Payment

Prepayment Penalty

YES

* As high as $3,240 if you pay off the loan during the first 2 years

Balloon Payment

NO

## Projected Payments

| Payment Calculation                      | Years 1-7  